# Algorithms 23-25: Regression via Pseudoinverse**Learning Objectives:**1. Formulate curve-fitting as an overdetermined system of linear equations $y = Xp$2. Use the Pseudoinverse $(X^T X)^{-1} X^T$ to find the optimal least-squares parameters $p$3. Extend the pseudoinverse to fit Polynomial, Multiple Linear, Power, Exponential, and Logistic models via algebraic transformation**Prerequisites:** Algorithms 15 (RREF/Matrix Inversion)**Estimated time:** 120 minutes**Textbook:** *Justin Skycak — Intro to Algorithms & Machine Learning* Chapters 23-25

In [ ]:
# Google Colab Setupimport mathimport matplotlib.pyplot as pltprint('Ready ✅')

---## Phase −1 — Theoretical Foundation### The Supervised Learning Problem> 📖 *From Algorithms Ch. 23:* Supervised learning consists of fitting functions to data sets. We want to come up with a mathematical model that predicts outputs for other inputs.Given data `[(0, 1), (2, 5), (4, 3)]`, we want to fit a line $y = mx + b$. We can plug each point into the equation:$1 = m(0) + b$$5 = m(2) + b$$3 = m(4) + b$This is an overdetermined system (3 equations, 2 unknowns). There is no perfect solution. In matrix form, $y = Xp$:$\begin{bmatrix} 1 \\ 5 \\ 3 \end{bmatrix} = \begin{bmatrix} 0 & 1 \\ 2 & 1 \\ 4 & 1 \end{bmatrix} \begin{bmatrix} m \\ b \end{bmatrix}$### The PseudoinverseBecause $X$ is tall and rectangular, it has no inverse. But if we multiply both sides by $X^T$, we project the problem into a square matrix space!$X^T y = X^T X p$Now, $X^T X$ is a square matrix (in this case $2 \times 2$). If its columns are independent, it is invertible!$p = (X^T X)^{-1} X^T y$This magical formula guarantees the mathematically optimal least-squares fit for *any* linear combination of functions!

### Comprehension Check ✅How would you set up the $X$ matrix to fit a quadratic model $y = ax^2 + bx + c$ to the same three points?<details><summary>💡 Answer</summary>Each row in $X$ would have 3 columns: $x^2$, $x$, and $1$.$\begin{bmatrix} 1 \\ 5 \\ 3 \end{bmatrix} = \begin{bmatrix} 0^2 & 0 & 1 \\ 2^2 & 2 & 1 \\ 4^2 & 4 & 1 \end{bmatrix} \begin{bmatrix} a \\ b \\ c \end{bmatrix}$</details>

---## Phase 0 — Math Foundation Practice### 🎯 Your Turn: Non-Linear TransformationsThe pseudoinverse can only solve models of the form $y = c_1 f_1(x) + c_2 f_2(x) + \dots$.If we want to fit a **Power Regression** $y = a \cdot x^b$, we must transform it.Take the natural logarithm of both sides and expand the right side using logarithm rules.What does the equation look like now? What are the new "variables" and "constants"?<details><summary>💡 Answer</summary>$\ln(y) = \ln(a \cdot x^b)$$\ln(y) = \ln(a) + b \ln(x)$Let $Y = \ln(y)$ and $X_{new} = \ln(x)$. The equation is $Y = b X_{new} + \ln(a)$, which is a straight line! We can solve for $b$ and $\ln(a)$ using the pseudoinverse on the transformed data.</details>

---## Phase 1 — Algorithm Construction### Step 1: The Matrix Class (Refresher)To do this, we need matrix multiplication, transpose, and inverse. Bring over your `Matrix` class from Algorithms 15. *(For the sake of time in this lab, you may use a simplified list-of-lists approach or a basic pure-Python matrix library if you already built one. Here is a starter.)*

In [ ]:
class Matrix:    def __init__(self, data):        self.data = data        self.rows = len(data)        self.cols = len(data[0])            def transpose(self):        return Matrix([[self.data[r][c] for r in range(self.rows)] for c in range(self.cols)])            def multiply(self, other):        result = [[sum(self.data[r][k] * other.data[k][c] for k in range(self.cols))                    for c in range(other.cols)] for r in range(self.rows)]        return Matrix(result)            # We need an inverse method! For 2x2, there is a simple formula:    def inverse_2x2(self):        a, b = self.data[0][0], self.data[0][1]        c, d = self.data[1][0], self.data[1][1]        det = a*d - b*c        return Matrix([[d/det, -b/det], [-c/det, a/det]])

### Step 2: Linear Regression SolverWrite a function `fit_linear(points)` that takes a list of `(x, y)` tuples and returns `(m, b)`.

In [ ]:
def fit_linear(points):    # 1. Construct X and Y matrices    # X should be [[x1, 1], [x2, 1], ...]    # Y should be [[y1], [y2], ...]    # YOUR CODE HERE        # 2. Compute X_T = X.transpose()    # 3. Compute X_T_X = X_T.multiply(X)    # 4. Compute X_T_X_inv = X_T_X.inverse_2x2()    # 5. Compute X_T_Y = X_T.multiply(Y)    # 6. Compute p = X_T_X_inv.multiply(X_T_Y)    # return (p.data[0][0], p.data[1][0]) # (m, b)    pass

### Step 3: Exponential Regression SolverWrite a function `fit_exponential(points)` that fits $y = a \cdot b^x$.Transform the data, use `fit_linear`, and then reverse the transformation to find $a$ and $b$.

In [ ]:
def fit_exponential(points):    # y = a * b^x  =>  ln(y) = ln(a) + x * ln(b)    # This is a line: Y = m*X + B, where Y = ln(y), m = ln(b), X = x, B = ln(a)        # 1. Transform the points: (x, y) -> (x, ln(y))    # YOUR CODE HERE        # 2. Call fit_linear on the transformed points to get (m, B)    # YOUR CODE HERE        # 3. Reverse the transformation:    # b = math.exp(m)    # a = math.exp(B)    # return (a, b)    pass

---## Phase 2 — Experimental Verification### Fitting and PlottingTest your algorithms on the dataset `[(1, 2), (2, 4), (3, 7), (4, 15), (5, 30)]`.Plot the data points as a scatter plot. Then, generate $X$ values from 1 to 5, compute the predicted $Y$ values using your exponential model, and plot the curve!

In [ ]:
# data = [(1, 2), (2, 4), (3, 7), (4, 15), (5, 30)]# a, b = fit_exponential(data)# print(f"Model: y = {a:.3f} * {b:.3f}^x")# xs = [p[0] for p in data]# ys = [p[1] for p in data]# plt.scatter(xs, ys, color='red', label="Data")# plot_xs = [1 + i*0.1 for i in range(41)]# plot_ys = [a * (b**x) for x in plot_xs]# plt.plot(plot_xs, plot_ys, color='blue', label="Exponential Fit")# plt.legend(); plt.grid(True); plt.show()

---## Phase 3 — Olympiad Extension### The Danger of Transformation> 📖 *From Algorithms Ch. 25:* If we transform $y = a x^b$ into $\ln(y) = \ln(a) + b \ln(x)$, the pseudoinverse finds the absolute mathematically optimal fit for the *transformed* equation. It does **NOT** guarantee the optimal fit for the *original* equation!This is because distances are warped by the logarithm. An error of $100$ at $y=1000$ looks like a tiny error in $\ln(y)$ space, while an error of $1$ at $y=2$ looks huge in $\ln(y)$ space! How could we solve for the parameters of $y = a x^b$ WITHOUT transforming the equation, ensuring we get the true optimal fit in the original space?<details><summary>💡 Sketch</summary>We cannot use the exact algebraic pseudoinverse formula, because the equation is strictly non-linear with respect to its parameters. Instead, we must formulate an error function $E(a, b) = \sum (a x_i^b - y_i)^2$, compute the partial derivatives $\frac{\partial E}{\partial a}$ and $\frac{\partial E}{\partial b}$, and use **Gradient Descent** (Chapter 27) to walk down the 3D error surface until we find the minimum!</details>---📚 **Next:** Algorithms 26 (K-Nearest Neighbors)